# Session 8 — Embeddings And Similarity

This notebook introduces embeddings as vector representations of meaning. The goal is to see how text can be turned into numbers that support similarity search and retrieval.

## Learning Goals

- create embeddings with the OpenAI SDK
- understand that embeddings are vectors, not generated text
- compare semantically similar and dissimilar texts with cosine similarity
- run a tiny semantic search example over a small text collection
- connect embeddings directly to the retrieval step used in the RAG notebook
- compare OpenAI embeddings with local and offline alternatives


In [1]:
import os

import numpy as np
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")
EMBEDDING_MODEL = "text-embedding-3-small"

print("OpenAI key present:", bool(OPENAI_API_KEY))
print("OpenAI org ID present:", bool(OPENAI_ORG_ID))
print("OpenAI project ID present:", bool(OPENAI_PROJECT_ID))
print("Embedding model:", EMBEDDING_MODEL)

OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True
Embedding model: text-embedding-3-small


In [2]:
def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def make_openai_client() -> OpenAI:
    return OpenAI(
        api_key=require_env("OPENAI_API_KEY"),
        organization=OPENAI_ORG_ID or None,
        project=OPENAI_PROJECT_ID or None,
    )


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def embed_texts(client: OpenAI, texts: list[str], model: str = EMBEDDING_MODEL) -> np.ndarray:
    response = client.embeddings.create(model=model, input=texts)
    return np.array([item.embedding for item in response.data], dtype="float32")

## 1. Create Embeddings

An embedding model converts text into a vector. The vector is not readable in the same way as generated text, but it places semantically similar texts closer together in vector space.

In [3]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client()
sample_texts = [
    "A cat is sleeping on the sofa.",
    "A kitten is napping on a couch.",
    "The stock market closed higher today.",
]

sample_embeddings = embed_texts(client, sample_texts)

display(Markdown(
    f"Created **{sample_embeddings.shape[0]}** embeddings with dimension **{sample_embeddings.shape[1]}** using `{EMBEDDING_MODEL}`."
))
sample_embeddings[:2, :8]

Created **3** embeddings with dimension **1536** using `text-embedding-3-small`.

array([[-0.05612183, -0.04626465, -0.03964233,  0.02174377,  0.03088379,
        -0.01455688,  0.00376892,  0.02615356],
       [-0.05770874, -0.0447998 , -0.04678345,  0.0435791 ,  0.07226562,
         0.01928711,  0.01064301,  0.02056885]], dtype=float32)

## 2. Compare Similarity

Cosine similarity gives a quick way to compare how close two embeddings are. Higher scores usually mean the texts are more semantically related.

In [4]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client()
pairs = [
    ("A dog is running in a park.", "A puppy is sprinting outside."),
    ("An employee can claim gym benefits.", "A health plan offers reimbursement for fitness classes."),
    ("A dog is running in a park.", "The database server needs a security patch."),
]

pair_lines = []
for left, right in pairs:
    pair_embeddings = embed_texts(client, [left, right])
    score = cosine_similarity(pair_embeddings[0], pair_embeddings[1])
    pair_lines.append(
        f"- **Similarity {score:.3f}**\n  - Text A: {left}\n  - Text B: {right}"
    )

display(Markdown("## Similarity results\n" + "\n".join(pair_lines)))

## Similarity results
- **Similarity 0.588**
  - Text A: A dog is running in a park.
  - Text B: A puppy is sprinting outside.
- **Similarity 0.566**
  - Text A: An employee can claim gym benefits.
  - Text B: A health plan offers reimbursement for fitness classes.
- **Similarity 0.070**
  - Text A: A dog is running in a park.
  - Text B: The database server needs a security patch.

## 3. Tiny Semantic Search

Retrieval starts with the same basic idea: embed the query, compare it to embedded texts, and return the closest matches.

In [5]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client()
documents = [
    "Employees can claim reimbursement for approved fitness memberships.",
    "Northwind Health Plus includes broader emergency coverage.",
    "Annual leave must be approved by a supervisor in advance.",
    "The office Wi-Fi password changes every quarter.",
]
query = "Which text is most relevant if I want support for gym or exercise costs?"

doc_embeddings = embed_texts(client, documents)
query_embedding = embed_texts(client, [query])[0]
scores = [cosine_similarity(query_embedding, doc_embedding) for doc_embedding in doc_embeddings]
ranked = sorted(zip(documents, scores), key=lambda item: item[1], reverse=True)

ranking_lines = [f"- **{score:.3f}** — {document}" for document, score in ranked]
display(Markdown(
    f"## Query\n{query}\n\n## Ranked results\n" + "\n".join(ranking_lines)
))

## Query
Which text is most relevant if I want support for gym or exercise costs?

## Ranked results
- **0.469** — Employees can claim reimbursement for approved fitness memberships.
- **0.187** — Northwind Health Plus includes broader emergency coverage.
- **0.115** — The office Wi-Fi password changes every quarter.
- **0.097** — Annual leave must be approved by a supervisor in advance.

## 4. Local And Offline Alternatives

OpenAI embeddings are the main teaching path in this session, but they are not the only option. Two useful alternatives are local embedding libraries and local OpenAI-compatible model servers.

- `sentence-transformers` runs directly in Python and works well when you want an offline baseline.
- Ollama can expose an OpenAI-compatible embeddings endpoint, so the same OpenAI SDK can talk to a local model server.


In [6]:
# Optional: local/offline embeddings with sentence-transformers
# This path does not use the OpenAI API at all.

from sentence_transformers import SentenceTransformer

local_model = SentenceTransformer('all-MiniLM-L6-v2')
local_embeddings = local_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
local_query_embedding = local_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
local_scores = [cosine_similarity(local_query_embedding, doc_embedding) for doc_embedding in local_embeddings]
local_ranked = sorted(zip(documents, local_scores), key=lambda item: item[1], reverse=True)

local_lines = [f"- **{score:.3f}** — {document}" for document, score in local_ranked]
display(Markdown(
    "## sentence-transformers results\n"
    "Using `all-MiniLM-L6-v2` as a local embedding model.\n\n"
    "## Ranked results\n" + "\n".join(local_lines)
))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\git\mdsiprojects\anlpaut2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MutazAbuGhazaleh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## sentence-transformers results
Using `all-MiniLM-L6-v2` as a local embedding model.

## Ranked results
- **0.363** — Employees can claim reimbursement for approved fitness memberships.
- **0.183** — Northwind Health Plus includes broader emergency coverage.
- **0.060** — The office Wi-Fi password changes every quarter.
- **0.048** — Annual leave must be approved by a supervisor in advance.

In practice, this is why the Session 8 RAG notebook can use `sentence-transformers` for retrieval even though the course is OpenAI-first. The core idea is unchanged: turn text into vectors, compare vectors, and retrieve the closest chunks.

Use `sentence-transformers` when you want a lightweight local baseline, reproducible offline experiments, or a retrieval pipeline that avoids API calls. Use OpenAI embeddings when you want the managed path that aligns most closely with the rest of the session.


In [7]:
# Optional: Ollama via the OpenAI SDK
# This is technically possible because Ollama exposes OpenAI-compatible endpoints,
# including an embeddings endpoint. You need Ollama running locally and a local
# embedding model pulled first, for example: `ollama pull nomic-embed-text`.

OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_EMBEDDING_MODEL = os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text')

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ollama_doc_embeddings = embed_texts(ollama_client, documents, model=OLLAMA_EMBEDDING_MODEL)
ollama_query_embedding = embed_texts(ollama_client, [query], model=OLLAMA_EMBEDDING_MODEL)[0]
ollama_scores = [cosine_similarity(ollama_query_embedding, doc_embedding) for doc_embedding in ollama_doc_embeddings]
ollama_ranked = sorted(zip(documents, ollama_scores), key=lambda item: item[1], reverse=True)

ollama_lines = [f"- **{score:.3f}** — {document}" for document, score in ollama_ranked]
display(Markdown(
    f"## Ollama via OpenAI SDK\nBase URL: `{OLLAMA_BASE_URL}`\nModel: `{OLLAMA_EMBEDDING_MODEL}`\n\n"
    "## Ranked results\n" + "\n".join(ollama_lines)
))


## Ollama via OpenAI SDK
Base URL: `http://localhost:11434/v1`
Model: `nomic-embed-text`

## Ranked results
- **0.632** — Employees can claim reimbursement for approved fitness memberships.
- **0.612** — Northwind Health Plus includes broader emergency coverage.
- **0.484** — Annual leave must be approved by a supervisor in advance.
- **0.483** — The office Wi-Fi password changes every quarter.

These three options map to different teaching and engineering goals:

- OpenAI embeddings: the main managed path for this course and the cleanest fit with the rest of the SDK examples.
- `sentence-transformers`: a pure Python offline alternative.
- Ollama with the OpenAI SDK: a local service-based alternative that preserves the OpenAI client pattern.


## Bridge To RAG

The next step is to apply this same pattern to chunked PDF documents instead of a tiny hand-written list. That is the core retrieval move inside a RAG pipeline: embed the chunks, embed the query, then fetch the nearest chunks before generating an answer.